In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# 1. Cloud path to the uploaded file in your volume
file_path = "/Volumes/workspace/default/Bank_loan/lending_club_loan_two.csv"

# 2. Ingest CSV with multiLine enabled
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .option("multiLine", "true") \
    .load(file_path)

# 3. Print baseline shape
initial_rows = df_raw.count()
initial_cols = len(df_raw.columns)

print("==================================================")
print("RAW DATASET INGESTED SUCCESSFULLY")
print(f"Total Raw Records: {initial_rows:,}")
print(f"Total Raw Columns: {initial_cols}")
print("==================================================")

# 4. Display Schema & Preview
df_raw.printSchema()
display(df_raw.limit(5))

RAW DATASET INGESTED SUCCESSFULLY
Total Raw Records: 396,030
Total Raw Columns: 27
root
 |-- loan_amnt: double (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: double (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)
 |-- dti: double (nullable = true)
 |-- earliest_cr_line: string (nullable = true)
 |-- open_acc: double (nullable = true)
 |-- pub_rec: double (nullable = true)
 |-- revol_bal: double (nullable = true)
 |-- revol_util: double (nullable = true)
 |-- total_acc: double (nullable = true)
 |-- ini

loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,title,dti,earliest_cr_line,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,pub_rec_bankruptcies,address
10000.0,36 months,11.44,329.48,B,B4,Marketing,10+ years,RENT,117000.0,Not Verified,Jan-2015,Fully Paid,vacation,Vacation,26.24,Jun-1990,16.0,0.0,36369.0,41.8,25.0,w,INDIVIDUAL,0.0,0.0,"0174 Michelle Gateway Mendozaberg, OK 22690"
8000.0,36 months,11.99,265.68,B,B5,Credit analyst,4 years,MORTGAGE,65000.0,Not Verified,Jan-2015,Fully Paid,debt_consolidation,Debt consolidation,22.05,Jul-2004,17.0,0.0,20131.0,53.3,27.0,f,INDIVIDUAL,3.0,0.0,"1076 Carney Fort Apt. 347 Loganmouth, SD 05113"
15600.0,36 months,10.49,506.97,B,B3,Statistician,< 1 year,RENT,43057.0,Source Verified,Jan-2015,Fully Paid,credit_card,Credit card refinancing,12.79,Aug-2007,13.0,0.0,11987.0,92.2,26.0,f,INDIVIDUAL,0.0,0.0,"87025 Mark Dale Apt. 269 New Sabrina, WV 05113"
7200.0,36 months,6.49,220.65,A,A2,Client Advocate,6 years,RENT,54000.0,Not Verified,Nov-2014,Fully Paid,credit_card,Credit card refinancing,2.6,Sep-2006,6.0,0.0,5472.0,21.5,13.0,f,INDIVIDUAL,0.0,0.0,"823 Reid Ford Delacruzside, MA 00813"
24375.0,60 months,17.27,609.33,C,C5,Destiny Management Inc.,9 years,MORTGAGE,55000.0,Verified,Apr-2013,Charged Off,credit_card,Credit Card Refinance,33.95,Mar-1999,13.0,0.0,24584.0,69.8,43.0,f,INDIVIDUAL,1.0,0.0,"679 Luna Roads Greggshire, VA 11650"


In [0]:
# Re-initialize initial_rows safety check in case kernel was restarted
try:
    initial_rows
except NameError:
    initial_rows = df_raw.count()

dq_log = []
current_count = initial_rows

# Rule 1: Semi-Structured Text Parsing and Extraction

df_clean_step1 = df_raw.withColumn(
    "term_months",
    F.regexp_extract(F.col("term"), r"(\d+)", 1).cast("int")
).withColumn(
    "borrower_state",
    F.regexp_extract(F.col("address"), r"([A-Z]{2})\s+\d{5}$", 1)
).withColumn(
    "borrower_zip",
    F.regexp_extract(F.col("address"), r"(\d{5})$", 1)
)

count_step1 = df_clean_step1.count()
dq_log.append((
    "Rule 1: Semi-structured text parsing",
    current_count,
    count_step1,
    current_count - count_step1,
    "Extracted integer term months; parsed multi-line address into State and Postal Code"
))
current_count = count_step1


# Rule 2: Target Variable Standardization

df_clean_step2 = df_clean_step1.withColumn(
    "is_default",
    F.when(F.col("loan_status").contains("Charged Off"), 1)
     .when(F.col("loan_status").contains("Fully Paid"), 0)
     .otherwise(None)
).filter(F.col("is_default").isNotNull())

count_step2 = df_clean_step2.count()
dq_log.append((
    "Rule 2: Target variable standardization",
    current_count,
    count_step2,
    current_count - count_step2,
    "Standardized loan_status into binary flag (1 = Charged Off, 0 = Fully Paid); removed invalid statuses"
))
current_count = count_step2


# Rule 3: Boundary and Outlier Filtering

df_clean_step3 = df_clean_step2.filter(
    (F.col("dti").isNotNull()) & (F.col("dti") >= 0) & (F.col("dti") <= 100) &
    (F.col("annual_inc").isNotNull()) & (F.col("annual_inc") > 0)
)

count_step3 = df_clean_step3.count()
dq_log.append((
    "Rule 3: Boundary & outlier filtering",
    current_count,
    count_step3,
    current_count - count_step3,
    "Eliminated impossible DTI records (>100% or negative) and zero/negative income records"
))
current_count = count_step3


# Rule 4: Missing Value Imputation

df_final_clean = df_clean_step3.na.fill({
    "mort_acc": 0.0,
    "pub_rec_bankruptcies": 0.0,
    "revol_util": 0.0,
    "emp_length": "Unknown",
    "emp_title": "Not Specified",
    "title": "Not Specified"
})

final_count = df_final_clean.count()
dq_log.append((
    "Rule 4: Missing attribute imputation",
    current_count,
    final_count,
    current_count - final_count,
    "Imputed null values in credit metrics with 0.0 and missing text fields with categorical placeholders"
))


# Building and Rendering the Audit Summary Table

dq_schema = ["Data Cleaning Rule Applied", "Input Row Count", "Output Row Count", "Rows Dropped", "Transformation Rationale"]
df_dq_summary = spark.createDataFrame(dq_log, dq_schema)

print(f"Final Cleaned Dataset Records: {final_count:,}")
display(df_dq_summary)

Final Cleaned Dataset Records: 396,022


Data Cleaning Rule Applied,Input Row Count,Output Row Count,Rows Dropped,Transformation Rationale
Rule 1: Semi-structured text parsing,396030,396030,0,Extracted integer term months; parsed multi-line address into State and Postal Code
Rule 2: Target variable standardization,396030,396030,0,"Standardized loan_status into binary flag (1 = Charged Off, 0 = Fully Paid); removed invalid statuses"
Rule 3: Boundary & outlier filtering,396030,396022,8,Eliminated impossible DTI records (>100% or negative) and zero/negative income records
Rule 4: Missing attribute imputation,396022,396022,0,Imputed null values in credit metrics with 0.0 and missing text fields with categorical placeholders


In [0]:
# Save the cleaned PySpark DataFrame as a managed Delta table in Unity Catalog
output_table_name = "workspace.default.lending_club_clean"

df_final_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(output_table_name)

print(f"Successfully saved cleaned data to Delta table: {output_table_name}")

Successfully saved cleaned data to Delta table: workspace.default.lending_club_clean


In [0]:
from pyspark.sql import functions as F

# 1. Load the clean Delta table
df_base = spark.read.table("workspace.default.lending_club_clean")

# 2. Add a unique surrogate primary key (loan_key) to every loan record
df_model = df_base.withColumn("loan_key", F.monotonically_increasing_id())


# A. Build & Save Fact Table: fact_loans

df_fact_loans = df_model.select(
    "loan_key",
    "loan_amnt",
    "installment",
    "int_rate",
    "annual_inc",
    "dti",
    "revol_bal",
    "revol_util",
    "total_acc",
    "mort_acc",
    "pub_rec_bankruptcies",
    "is_default"
)

df_fact_loans.write.format("delta").mode("overwrite").saveAsTable("workspace.default.fact_loans")
print("Successfully created Delta table: workspace.default.fact_loans")


# B. Build & Save Dimension Table: dim_borrower

df_dim_borrower = df_model.select(
    "loan_key",
    "borrower_state",
    "borrower_zip",
    "emp_length",
    "emp_title",
    "home_ownership"
)

df_dim_borrower.write.format("delta").mode("overwrite").saveAsTable("workspace.default.dim_borrower")
print("Successfully created Delta table: workspace.default.dim_borrower")


# C. Build & Save Dimension Table: dim_loan_details

df_dim_loan_details = df_model.select(
    "loan_key",
    "term_months",
    "grade",
    "sub_grade",
    "purpose",
    "verification_status",
    "initial_list_status"
)

df_dim_loan_details.write.format("delta").mode("overwrite").saveAsTable("workspace.default.dim_loan_details")
print("Successfully created Delta table: workspace.default.dim_loan_details")


print("STAR SCHEMA MODELING COMPLETE!")


Successfully created Delta table: workspace.default.fact_loans
Successfully created Delta table: workspace.default.dim_borrower
Successfully created Delta table: workspace.default.dim_loan_details
STAR SCHEMA MODELING COMPLETE!


In [0]:
CAT = "workspace.default"

# Integrity checks
fl = spark.read.table(f"{CAT}.fact_loans")
n = spark.read.table(f"{CAT}.lending_club_clean").count()

# Assertions
assert fl.count() == n and fl.select("loan_key").distinct().count() == n
assert fl.where("borrower_key IS NULL OR loan_detail_key IS NULL OR geography_key IS NULL OR date_key IS NULL").count() == 0

print("==================================================")
print("SUCCESS: Star Schema integrity checks passed!")
print(f"Cleaned Row Count: {n:,}")
print(f"Fact Table Row Count: {fl.count():,}")
print("Zero null foreign keys detected. Surrogate keys are unique.")
print("==================================================")

SUCCESS: Star Schema integrity checks passed!
Cleaned Row Count: 396,022
Fact Table Row Count: 396,022
Zero null foreign keys detected. Surrogate keys are unique.
